In [ ]:
parcels_of_interest = {
    'theory' : [133,172,192,284,339,395],
    'data_driven' : [9,67,176,183,231,242,341,343,348,380,386,389,391]
}
# theory_parcels = [133,172,192,284,339,395]
# # Data driven seed values
# data_driven_parcels = [9,67,176,183,231,242,341,343,348,380,386,389,391]

# important demographics: age, years experience, GPA, Code_more_structured, 
# correct, complete, code_correct, code_complete
variables = {
    'GPA': 'GPA',
    'Age': 'age',
    'Years of Experience': 'years_experience',
    # 'Code Editing' : 'code_editing_rate',
    # 'Prose Editing' : 'prose_editing_rate',
    # 'Code Total Keystrokes' : 'code_total_keystrokes',
    # 'Prose Total Keystrokes' : 'prose_total_keystrokes',
    # 'Code More Structured' : 'Code_more_structured', 
    'Correct' : 'correct',
    'Complete' : 'complete',
    'Code Correct' : 'code_correct',
    'Code Complete' : 'code_complete'
}

demo_data = pd.read_csv("/home/zachkaras/fmri_model/master-survey-data.csv")
demo_data['years_experience'] = demo_data['semesters_experience'] / 2
# demo_data['code_more_structured'] = demo_data['Code_more_structured'].map({'X': 0, 'Y': 1})
demo_data = demo_data.set_index('id')
print(demo_data[['age', 'GPA', 'years_experience']].describe())


def find_parcel_voxels(parcel_num):
    parcel_idx = list((np.where(parcel_nums == parcel_num))[0])
    print(len(parcel_idx), parcel_idx)


from scipy import stats

def find_cutoff(vec, threshold=10**4):
    copy = vec.copy()
    copy.sort()
    return copy[-threshold - 1]

best_models = ['code-deepseek_6b-ndelays_4-look_ahead_by_10',
               'code-codegemma_7b-ndelays_4-look_ahead_by_10',
               'code-codegemma_7b-ndelays_16-look_ahead_by_0',
               'prose-codegemma_7b-ndelays_10-look_ahead_by_5',
               'prose-deepseek_6b-ndelays_20-look_ahead_by_10']

participant_base_path = "/data/zachkaras/fmri_model_data/ridge_regression_pca_params"
participants = os.listdir(participant_base_path)

In [ ]:
records = []

for m in best_models:
    parts = m.split('-')
    task, model, delays, look_ahead = parts[0], parts[1], parts[2], parts[3]
    pattern = f"{model}-{task}-{look_ahead}-{delays}"

    for p in participants:
        files = os.listdir(f"{participant_base_path}/{p}")
        best_files = [f for f in files if re.search(pattern, f) and 'correlations' in f]

        # Accumulate per-layer means for each parcel, then average across layers
        layer_vals = defaultdict(list)  # (approach, parcel) -> [layer_mean, ...]

        for bf in best_files:
            with open(f"{participant_base_path}/{p}/{bf}", 'rb') as f:
                corrs = pickle.load(f)
            z_corrs = np.arctanh(corrs)

            for approach, parcels in parcels_of_interest.items():
                for parcel in parcels:
                    layer_vals[(approach, parcel)].append(
                        np.mean(find_parcel_voxels(parcel, z_corrs))
                    )

        # One row per participant × model × parcel, averaged across layers
        for (approach, parcel), vals in layer_vals.items():
            records.append({
                'model': m,
                'participant': int(p),
                'approach': approach,
                'parcel': parcel,
                'mean_z_corr': np.mean(vals),
            })

df = pd.DataFrame(records)
df = df.merge(demo_data[list(variables.values())], left_on='participant', right_index=True)

Then stats are just:

from scipy import stats

stat_records = []
for (model, approach, parcel), group in df.groupby(['model', 'approach', 'parcel']):
    for label, col in variables.items():
        r, p = stats.spearmanr(group['mean_z_corr'], group[col], nan_policy='omit')
        stat_records.append({
            'model': model, 'approach': approach, 'parcel': parcel,
            'variable': label, 'r': r, 'p': p
        })

stats_df = pd.DataFrame(stat_records)